# M18 — Calibrate probabilities, challenge explanations, preserve the evidence

<!-- paper-first -->
### Begin with the paper question

**Read or revisit [PM03](../../curriculum/papers/modeling.md#pm03), [PM04](../../curriculum/papers/modeling.md#pm04).** Use the assigned first-pass sections in the guide; if you already read them, return only to the relevant figure or claim. Do this before the technical explanation below.

**Motivation:** Could a predictor discriminate well while giving poorly calibrated probabilities or misleading explanations?

Write a two-sentence prediction and one thing you cannot yet explain. Ask your AI tutor to locate evidence in the supplied paper and distinguish it from inference. A paper link motivates this question; it does not mean the paper uses every method demonstrated here.

**After the experiment:** revisit your prediction in the [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md). Explain one mechanism you now understand, cite a result from this notebook, and name a paper claim this exercise still cannot test. Keep a small demonstration distinct from a reproduction of the study.
<!-- /paper-first -->

**Original guided lab · 75–100 minutes.** Read 20 min, predict/code 35 min, failure investigation 20 min, explain and transfer 15 min. Run all cells in order in a fresh kernel. All executed data are synthetic unless explicitly stated. No network, GPU, or external dataset is required.

A final model report should distinguish discrimination, probability quality, decision behavior, robustness, and explanation. These are related but different claims. ROC AUC measures ranking; precision and recall depend on a threshold and prevalence; a calibration curve compares stated probabilities with observed frequencies. A probability can be confidently wrong even when the ranking is excellent.

Calibration is itself a fitted transformation. A sigmoid calibration model learns how to map a score into a probability using a calibration set that was not used to train the score model. The final test set must remain untouched by fitting and threshold selection. Crossvalidated calibration can use data more efficiently, but its folds must still respect participants, sites, and time dependencies. Calibration learned at one prevalence or setting may not transfer unchanged to another.

The first experiment constructs probabilities from a known logistic data generator, then deliberately exaggerates its scores. A scalar logistic calibrator is fitted on a separate calibration portion. The held-out Brier score should improve while AUC remains the same because a positive monotonic mapping preserves ranking. This clean result relies on the constructed setting; calibration cannot create missing discriminatory information or repair a scanner-induced shortcut.

Explanation tools must also be tied to a question. Permutation importance measures the change in a selected predictive score when a feature is disrupted under a specified procedure. Correlated features can substitute for one another, making an individually permuted feature appear unimportant; unrealistic permutations can create out-of-distribution combinations. Saliency and attention plots have different meanings and should be subjected to randomization and sensitivity checks. None of these methods turns prediction into causal evidence.

We fit a simple regression with one informative feature and inspect held-out permutation importance. The test is intentionally easy, so the planted feature should stand out. The point is to understand which score is being disrupted, not to claim that the method identifies brain mechanisms. Real feature perturbations should preserve the dependencies relevant to the claim or explicitly acknowledge the departure.

Reproducibility requires more than a seed. Preserve input identifiers, splits, transformation parameters, code and package versions, model checkpoints, metrics, and decisions made after seeing outcomes. A checksum can establish that the hashed arrays match; it does not establish that they were appropriate or independent. Rerunning the same data verifies a computation. Independent replication and evaluation at another site ask whether the result survives new evidence. Ask AI to separate these outcomes in its report and never write a successful test result that was not actually observed.

## Transformation contract

Scores → calibration-set sigmoid fit → held-out probabilities and metrics. Feature perturbation → score change → importance summary. Reproducibility record preserves the executed lineage; it cannot restore unrecorded scientific decisions or turn repeated computation into replication.

## Ask your AI tutor

```text
Explain this notebook one transformation at a time.
Before each cell ask me to predict shapes, units, and a check.
Give edits in executable cells of at most 20 lines.
Keep the prescribed split, random seed, and tests intact.
Distinguish generated suggestions from executed results.
After the failure experiment, ask me to explain the mechanism.
```

In [1]:
import numpy as np
from scipy.special import expit
from sklearn.linear_model import LogisticRegression,Ridge
from sklearn.metrics import brier_score_loss,roc_auc_score
from sklearn.inspection import permutation_importance
rng=np.random.default_rng(418)
s=rng.normal(size=2200);y=rng.binomial(1,expit(s));raw=expit(4*s)
calibration=np.arange(800);test=np.arange(800,2200)
calibrator=LogisticRegression(C=1e4).fit((4*s[calibration])[:,None],y[calibration])
p=calibrator.predict_proba((4*s[test])[:,None])[:,1]
print('Raw/calibrated Brier:',brier_score_loss(y[test],raw[test]),brier_score_loss(y[test],p))
print('Raw/calibrated AUC:',roc_auc_score(y[test],raw[test]),roc_auc_score(y[test],p))
assert brier_score_loss(y[test],p)<brier_score_loss(y[test],raw[test])
assert np.isclose(roc_auc_score(y[test],p),roc_auc_score(y[test],raw[test]))


Raw/calibrated Brier: 0.2475395188887777 0.20482820360437995
Raw/calibrated AUC: 0.7474149247393489 0.7474149247393489


In [2]:
X=rng.normal(size=(600,5));target=3*X[:,0]+rng.normal(0,.4,600)
model=Ridge(alpha=1).fit(X[:350],target[:350])
importance=permutation_importance(model,X[350:],target[350:],n_repeats=8,random_state=2)
print('Held-out permutation importance:',importance.importances_mean)
assert importance.importances_mean[0]>1 and np.max(importance.importances_mean[1:])<.05
import hashlib,json,sklearn
record={'data_sha256':hashlib.sha256(X.tobytes()).hexdigest(),'train_rows':350,'test_rows':250,
        'seed':418,'sklearn':sklearn.__version__,'metric':'R² drop on held-out rows',
        'real_MRI':False,'independent_site_validation':'not performed'}
print(json.dumps(record,indent=2))
assert record['data_sha256']==hashlib.sha256(X.copy().tobytes()).hexdigest()


Held-out permutation importance: [ 1.90288362e+00 -3.62100760e-05  1.80821226e-04 -1.64211530e-05
  6.89680808e-05]
{
  "data_sha256": "6dbd1fff7f433b828e4834597ffec4627e58e82977d29a22f4144920f8cf05fc",
  "train_rows": 350,
  "test_rows": 250,
  "seed": 418,
  "sklearn": "1.9.1",
  "metric": "R\u00b2 drop on held-out rows",
  "real_MRI": false,
  "independent_site_validation": "not performed"
}


## Deliberate failure and repair

Exaggerated scores preserve ranking and degrade probability quality. Repair with a separately fitted calibration procedure and evaluate it on untouched data. Do not select a threshold by repeatedly checking the test outcomes. A second failure is calling the important synthetic feature a causal biomarker; the experiment only demonstrates predictive dependence under its generator.

## Your investigation

Create a report with baseline, split, discrimination, probability error, threshold policy, subgroup/site assessment, and limitations. Propose a saliency randomization check and explain what failure would mean. Change one input element and verify that the checksum changes; then state three scientific errors the checksum could not detect.

## Transfer to real neuroimaging

Apply participant-aware resampling, independent-site assessment, and subgroup reporting to a real cohort. Preserve model and preprocessing revisions, and separate exploratory explanation from confirmatory evidence. No clinical threshold, biomarker, or real-model validation is established by these synthetic checks.

**Primary teaching sources, pinned where hosted on GitHub:**

- [BrainIAK: model evaluation and selection](https://github.com/brainiak/brainiak-tutorials/blob/fb62ede943d9694fe703aee0df5f43ecf5558415/tutorials/05-classifier-optimization.ipynb)
- [scikit-learn probability calibration](https://scikit-learn.org/stable/modules/calibration.html)
- [scikit-learn permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html)
- [Adebayo et al.: Sanity Checks for Saliency Maps](https://arxiv.org/abs/1810.03292)

Pinned upstream tutorials are a separate assignment; they have **not been executed** by this core lab. They may require data downloads, specialist dependencies, unfinished student cells, and additional compute.

## Exit questions and answer key

1. Can AUC stay fixed while calibration changes? **Yes; monotonic score transformations preserve ranking.**
2. Does a repeatable checksum prove valid generalization? **No; it identifies data, not the validity of their use or the target population.**

### Return to the research question

Reopen [PM03](../../curriculum/papers/modeling.md#pm03), [PM04](../../curriculum/papers/modeling.md#pm04) and your initial two-sentence prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure/section locator. State what this small exercise still cannot establish about the published result.
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Reuse this entry in the A2 portfolio when relevant; a separate report is unnecessary.
